In [37]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce RTX 5070 Ti


In [38]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from transformers import BertTokenizer, BertForSequenceClassification

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from tqdm import tqdm

In [39]:
DATA_DIR = "../datasets/processed"

def load_all_datasets():
    paths = {
        'imdb': os.path.join(DATA_DIR, 'imdb.csv'),
        'rotten': os.path.join(DATA_DIR, 'rotten.csv'),
        'amazon': os.path.join(DATA_DIR, 'amazon.csv'),
        'yelp': os.path.join(DATA_DIR, 'yelp.csv')
    }

    datasets = {}

    for name, path in paths.items():
        if not os.path.exists(path):
            raise FileNotFoundError(f"Nie znaleziono pliku: {path}")

        df = pd.read_csv(path)
        df = df[['text', 'label']].dropna()
        df['label'] = df['label'].astype(int)
        datasets[name] = df

    return datasets

datasets = load_all_datasets()

In [40]:
def compute_max_len_bert(datasets, cap=256):
    max_lengths = {}

    for name, df in datasets.items():
        lengths = df['text'].apply(lambda x: len(str(x).split()))
        p95 = int(np.percentile(lengths, 95))
        max_len = min(p95, cap)

        max_lengths[name] = {'recommended_max_len': max_len}
        print(f"{name}: {max_len}")

    return max_lengths

max_lengths = compute_max_len_bert(datasets)

imdb: 256
rotten: 37
amazon: 248
yelp: 256


In [41]:
MODEL_NAME = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

In [42]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.encodings = tokenizer(
            list(texts),
            padding="max_length",
            truncation=True,
            max_length=max_len,
            return_tensors="pt"
        )
        # WAŻNE: LongTensor dla klasyfikacji z num_labels=2
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

In [43]:
def build_bert():
    model = BertForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2
    )
    return model.to(device)


In [44]:
def train_model(model, train_loader, val_loader, epochs=3):
    optimizer = AdamW(model.parameters(), lr=2e-5)

    best_val_loss = float("inf")
    patience = 2
    patience_counter = 0

    history_rows = []

    for epoch in range(epochs):
        print(f"\nEpoch {epoch + 1}/{epochs}")

        # TRAIN
        model.train()
        total_loss = 0.0

        for batch in tqdm(train_loader):
            batch = {k: v.to(device) for k, v in batch.items()}

            optimizer.zero_grad(set_to_none=True)
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_train_loss = total_loss / max(1, len(train_loader))

        # VALIDATION
        model.eval()
        val_loss = 0.0

        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                outputs = model(**batch)
                val_loss += outputs.loss.item()

        avg_val_loss = val_loss / max(1, len(val_loader))

        print(f"Train loss: {avg_train_loss:.4f} | Val loss: {avg_val_loss:.4f}")

        history_rows.append({
            "epoch": epoch + 1,
            "train_loss": avg_train_loss,
            "val_loss": avg_val_loss
        })

        # Early stopping
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping")
                break

    return pd.DataFrame(history_rows)

In [45]:
def predict(model, loader):
    model.eval()
    preds = []

    with torch.no_grad():
        for batch in loader:
            # do inference bez labels
            inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}

            outputs = model(**inputs)
            logits = outputs.logits

            pred = torch.argmax(logits, dim=1)
            preds.extend(pred.cpu().numpy().tolist())

    return np.array(preds)

In [46]:
def run_ood_experiment_bert(
    datasets,
    max_lengths,
    epochs=3,
    batch_size=16,
    n_splits=5
):
    results = []
    training_logs = []

    domains = list(datasets.keys())

    for train_domain in domains:
        print("\n" + "=" * 80)
        print(f"TRAIN DOMAIN: {train_domain}")
        print("=" * 80)

        df = datasets[train_domain].reset_index(drop=True)
        max_len = max_lengths[train_domain]["recommended_max_len"]
        print("MAX_LEN:", max_len)

        X_text = df["text"].values
        y = df["label"].values

        kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

        for fold_id, (train_idx, val_idx) in enumerate(kf.split(X_text), 1):
            print(f"\nFold {fold_id}/{n_splits}")

            train_dataset = TextDataset(
                X_text[train_idx], y[train_idx], tokenizer, max_len
            )
            val_dataset = TextDataset(
                X_text[val_idx], y[val_idx], tokenizer, max_len
            )

            train_loader = DataLoader(
                train_dataset,
                batch_size=batch_size,
                shuffle=True,
                pin_memory=(device.type == "cuda")
            )
            val_loader = DataLoader(
                val_dataset,
                batch_size=batch_size,
                shuffle=False,
                pin_memory=(device.type == "cuda")
            )

            model = build_bert()

            fold_history = train_model(model, train_loader, val_loader, epochs=epochs)
            for _, row in fold_history.iterrows():
                training_logs.append({
                    "train_domain": train_domain,
                    "fold": fold_id,
                    "epoch": int(row["epoch"]),
                    "train_loss": float(row["train_loss"]),
                    "val_loss": float(row["val_loss"])
                })

            # =========================
            # IN-DOMAIN
            # =========================
            y_val = y[val_idx]
            preds = predict(model, val_loader)

            acc = accuracy_score(y_val, preds)
            prec = precision_score(y_val, preds, zero_division=0)
            rec = recall_score(y_val, preds, zero_division=0)
            f1 = f1_score(y_val, preds, zero_division=0)

            results.append({
                "train_domain": train_domain,
                "test_domain": train_domain,
                "fold": fold_id,
                "accuracy": acc,
                "precision": prec,
                "recall": rec,
                "f1": f1,
                "eval_type": "IND"
            })

            print(f"IND | F1: {f1:.4f} | Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f}")

            # =========================
            # OOD
            # =========================
            for test_domain, test_df in datasets.items():
                if test_domain == train_domain:
                    continue

                X_test_text = test_df["text"].values
                y_test = test_df["label"].values

                test_dataset = TextDataset(
                    X_test_text, y_test, tokenizer, max_len
                )
                test_loader = DataLoader(
                    test_dataset,
                    batch_size=batch_size,
                    shuffle=False,
                    pin_memory=(device.type == "cuda")
                )

                preds = predict(model, test_loader)

                acc = accuracy_score(y_test, preds)
                prec = precision_score(y_test, preds, zero_division=0)
                rec = recall_score(y_test, preds, zero_division=0)
                f1 = f1_score(y_test, preds, zero_division=0)

                results.append({
                    "train_domain": train_domain,
                    "test_domain": test_domain,
                    "fold": fold_id,
                    "accuracy": acc,
                    "precision": prec,
                    "recall": rec,
                    "f1": f1,
                    "eval_type": "OOD"
                })

                print(f"OOD {train_domain} → {test_domain} | F1: {f1:.4f} | Acc: {acc:.4f}")

            if device.type == "cuda":
                torch.cuda.empty_cache()

    return pd.DataFrame(training_logs), pd.DataFrame(results)

In [47]:
training_logs_bert, ood_results_bert = run_ood_experiment_bert(
    datasets=datasets,
    max_lengths=max_lengths,
    epochs=3,
    batch_size=16,
    n_splits=5
)



TRAIN DOMAIN: imdb
MAX_LEN: 256

Fold 1/5


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 16555.59it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpo


Epoch 1/3


100%|██████████| 2500/2500 [05:00<00:00,  8.33it/s]


Train loss: 0.2502 | Val loss: 0.1959

Epoch 2/3


 52%|█████▏    | 1299/2500 [02:36<02:24,  8.33it/s]


KeyboardInterrupt: 

In [ ]:
SAVE_DIR = "../minilm/results"
os.makedirs(SAVE_DIR, exist_ok=True)

training_logs_bert.to_csv(os.path.join(SAVE_DIR, "bert_training_logs.csv"), index=False)
ood_results_bert.to_csv(os.path.join(SAVE_DIR, "bert_ood_results.csv"), index=False)

print("Saved:")
print(" - bert_training_logs.csv")
print(" - bert_ood_results.csv")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

pivot = ood_results_bert.groupby(
    ["train_domain", "test_domain"]
)["f1"].mean().unstack()

plt.figure(figsize=(8, 6))
sns.heatmap(pivot, annot=True, cmap="viridis")
plt.title("BERT OOD F1")
plt.show()